In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("q1_working.csv")

In [2]:
## mapping sport according to arch_type
sport_mapping = {
    "Archery": "Precision",
    "Shooting": "Precision",
    "Golf": "Precision",

    "Boxing": "Combat",
    "Wrestling": "Combat",
    "Judo": "Combat",

    "Badminton": "Racket Sports",
    "Tennis": "Racket Sports",
    "Table Tennis": "Racket Sports",

    "Hockey": "Team Sports",
    "Football": "Team Sports",
    "Basketball": "Team Sports",

    "Rowing": "Endurance",

    "Weightlifting": "Speed & Power",

    "Gymnastics": "Technical / Mixed",
    "Diving": "Technical / Mixed",
    "Sailing": "Technical / Mixed",
    "Equestrianism": "Technical / Mixed"
}

df["Archetype"] = df["Sport"].map(sport_mapping)

In [3]:
## sports where event level classification is required

remaining = df[df["Archetype"].isna()]

print(remaining["Sport"].value_counts())

display(
    remaining[["Sport", "Event"]]
    .drop_duplicates()
    .sort_values(["Sport", "Event"])
)


Sport
Athletics    179
Swimming      13
Cycling        3
Name: count, dtype: int64


,Sport,Event
269,Athletics,"Men's 1,500 metres"
92,Athletics,"Men's 10,000 metres"
197,Athletics,Men's 100 metres
600,Athletics,Men's 110 metres Hurdles
459,Athletics,Men's 20 kilometres Walk
423,Athletics,Men's 200 metres
118,Athletics,Men's 4 x 100 metres Relay
136,Athletics,Men's 4 x 400 metres Relay
77,Athletics,Men's 400 metres
390,Athletics,Men's 400 metres Hurdles


In [4]:
## special cases are Athletics , swimming,cycling
print("Classified records:", df["Archetype"].notna().sum())
print("Unclassified records:", df["Archetype"].isna().sum()) # unclassified ones

print("\nCurrent archetype counts:")
print(df["Archetype"].value_counts(dropna=False))

Classified records: 640
Unclassified records: 195

Current archetype counts:
Archetype
Team Sports          245
NaN                  195
Precision            144
Combat               115
Racket Sports         76
Technical / Mixed     28
Speed & Power         22
Endurance             10
Name: count, dtype: int64


In [5]:
## classifing the un classified

import re # have to use this to enable regex but forgot to mention in the report(before the begin of project)

def classify_event(row):
    sport = row["Sport"]
    event = row["Event"]

    # already classified using sport_mapping
    if pd.notna(row["Archetype"]):
        return row["Archetype"]

    if sport == "Athletics":

        # long-distance running and walking
        if any(x in event for x in [
            "800 metres", "1,500 metres",
            "3,000 metres", "5,000 metres",
            "10,000 metres", "Marathon",
            "kilometres Walk"
        ]):
            return "Endurance"

        # combined events
        if "Decathlon" in event or "Heptathlon" in event:
            return "Technical / Mixed"

        # sprints, jumps, throws and hurdles
        return "Speed & Power"

    if sport == "Swimming":
        # All swimming events present are 50-200 m
        return "Speed & Power"

    if sport == "Cycling":
        if "Sprint" in event:
            return "Speed & Power"
        return "Endurance"

    return "Unclassified"


df["Archetype"] = df.apply(classify_event, axis=1)

In [6]:
## verifing classifications
print(df["Archetype"].value_counts())

print("\nUnclassified records:")
print(df[df["Archetype"] == "Unclassified"].shape[0])

Archetype
Team Sports          245
Speed & Power        157
Precision            144
Combat               115
Racket Sports         76
Endurance             61
Technical / Mixed     37
Name: count, dtype: int64

Unclassified records:
0


In [7]:
## athlete belonging to different archtype in the same year
archetype_counts = (
    df.groupby(["ID", "Year"])["Archetype"]
    .nunique()
)

multiple_roles = archetype_counts[archetype_counts > 1]

print("Athlete-year groups with multiple archetypes:",
      len(multiple_roles))

display(multiple_roles)

Athlete-year groups with multiple archetypes: 5


ID      Year
512     1984    2
        1988    2
72165   2016    2
111176  1964    2
111234  1960    2
Name: Archetype, dtype: int64

In [8]:
## keeping the above athelete is necessary it's kind of like it is a different observation in real time 
## so we are gonna keep them but let's track them too if time permits then I would do seperate analysis 
## without them showing the impact
df_q1 = (
    df.sort_values(["ID", "Year", "Sport", "Event"])
      .drop_duplicates(
          subset=["ID", "Year", "Archetype"]
      )
      .copy()
)

print("Original records:", len(df))
print("Q1 records:", len(df_q1))
print("Unique athlete-years:",
      df_q1[["ID", "Year"]].drop_duplicates().shape[0])

Original records: 835
Q1 records: 737
Unique athlete-years: 732


In [9]:
df_q1.to_csv("q1_working.csv", index=False)

print("Saved q1_cleaned.csv")
print(df_q1.shape)

display(df_q1.head())

Saved q1_cleaned.csv
(737, 14)


,ID,Name,Sex,Age,Height,Weight,Year,City,Sport,Event,Medal,Clean_Name,BMI,Archetype
1,512,Shiny Kurisingal Abraham-Wilson,F,19.0,167.0,53.0,1984,Los Angeles,Athletics,Women's 4 x 400 metres Relay,No Medal,Shiny Kurisingal Abraham-Wilson,19.003908,Speed & Power
0,512,Shiny Kurisingal Abraham-Wilson,F,19.0,167.0,53.0,1984,Los Angeles,Athletics,Women's 800 metres,No Medal,Shiny Kurisingal Abraham-Wilson,19.003908,Endurance
3,512,Shiny Kurisingal Abraham-Wilson,F,23.0,167.0,53.0,1988,Seoul,Athletics,Women's 4 x 400 metres Relay,No Medal,Shiny Kurisingal Abraham-Wilson,19.003908,Speed & Power
2,512,Shiny Kurisingal Abraham-Wilson,F,23.0,167.0,53.0,1988,Seoul,Athletics,Women's 800 metres,No Medal,Shiny Kurisingal Abraham-Wilson,19.003908,Endurance
4,512,Shiny Kurisingal Abraham-Wilson,F,27.0,167.0,53.0,1992,Barcelona,Athletics,Women's 800 metres,No Medal,Shiny Kurisingal Abraham-Wilson,19.003908,Endurance
